# Fine-tune VideoMAE trên clip cắt quanh từng người (person-crop)

Bản này sửa notebook cũ theo những gì debug đã cho thấy:

| Vấn đề cũ | Cách sửa |
|---|---|
| Train trên cả khung hình, chạy thật lại đưa crop người vào (lệch phân phối) | Train **đúng trên crop người**, cùng hàm cắt với pipeline thật (`crops_lite.py`) |
| Clip dài ~10 s, lúc chạy dùng cửa sổ 3 s | `clip_sec` cố định 3–5 s, lúc chạy dùng đúng giá trị đó |
| Cả video bị gán một nhãn | Bạn ghi **khoảng thời gian** hành vi trong CSV, phần còn lại là normal |
| Chia clip ngẫu nhiên → rò rỉ | Chia **theo video nguồn**; video test cố định, không bao giờ vào train |
| Chỉ đo theo cửa sổ | Đo thêm **theo video**: bắt được bao nhiêu vụ, báo nhầm bao nhiêu lần mỗi phút |
| Ngưỡng chọn theo vài video, điểm nhiễu | Chọn (ngưỡng, làm mượt, K liên tiếp) trên **val**, lưu vào bundle |

### Bản sửa ngày 2026-11-17 (xem `docs/phan_tich_notebooks.md`)

| Lỗi trong lần chạy trước | Sửa |
|---|---|
| Test tự chọn không có video trộm (loại hết video UCF có nhãn vì nằm trong `annotations_fixed`) | Bỏ điều kiện đó; mặc định giữ 2 video UCF trộm làm test; cảnh báo nếu test không có cửa sổ dương |
| 3 cặp clip Kaggle trùng hệt nhau (normal-44/52, 50/51, 55/56), UCF Normal_936/937 | Loại video trùng (so kích thước + 2 MB đầu) |
| Clip Kaggle cùng camera/cảnh/người diễn rơi vào cả train và val | Chia train/val theo **nhóm cảnh** (dHash 3 khung), cả nhóm vào một bên |
| Kaggle 640x480, UCF 320x240: độ phân giải là "manh mối" nguồn | Tăng cường: thu nhỏ rồi phóng lại (`LOWRES_P`) |
| Overfit (loss 0.42 -> 0.07, val đứng yên) | Đóng băng embedding + `FREEZE` block đầu (mặc định 4), 6 epoch |
| Tăng cường không seed | Seed theo chỉ số mẫu |

## File annotation `annotations.csv` (bạn tự xem video và ghi)

```
video,start,end,label
shop_lifter_1.mp4,12.0,17.5,shoplifting
shop_lifter_2.mp4,40.0,46.0,shoplifting
normal_aisle_1.mp4,0,0,normal
```

- Mỗi dòng `shoplifting` là **một khoảng giây** trong đó người đó đang giấu hàng (đưa hàng vào túi/áo). Một video có thể có nhiều dòng.
- **Rất nên** thêm hai cột `px,py` khi trong cảnh có nhiều người: toạ độ pixel của video (x, y) của một điểm nằm trên **người trộm** ở giây `start` (mở khung hình đó, đọc toạ độ chuột). Khi có điểm này, những người đứng cạnh trong lúc đó được dạy là **không trộm** (ví dụ âm khó); không có thì họ chỉ bị bỏ qua.

  ```
  video,start,end,label,px,py
  shop_lifter_1.mp4,12.0,17.5,shoplifting,640,380
  ```
- Có thể ghi theo **khung hình** thay cho giây: cột `start_frame,end_frame` (để trống `start,end`); notebook đổi sang giây bằng fps thật của video. Bộ UCF-Crime dùng kiểu này.
- Có thể có **nhiều file** `annotations*.csv` (vd. `annotations.csv` của bạn và `annotations_ucf.csv`); notebook gộp tất cả.
- Notebook đọc video từ **mọi dataset** bạn đã Add Input (vd. Kaggle "Shoplifting Video Dataset" + dataset UCF của bạn). `VIDEO_DIRS=ten1,ten2` để chỉ lấy một số.
- Video trong thư mục tên `shoplifting/` mà **không** có trong CSV: nếu ngắn (≤ 30 s) thì cả clip là trộm; nếu dài hơn thì bị **bỏ qua** (không đoán nhãn).
- Video chỉ có dòng `normal` (hoặc không có dòng nào) thì toàn bộ là normal.
- **Phải tự xem video để ghi giờ.** Đừng nhờ AI "mô tả" video rồi chép, AI không xem được hình.
- Cần video normal thật (mua sắm bình thường, cúi nhặt, bỏ vào giỏ) và nhiều video khác nhau, không chỉ 1–2 video.
- Mặc định `MASK=1`: trong mỗi crop, những người khác được tô xám để điểm của người này không bị kéo theo người đứng cạnh. Đặt `MASK=0` nếu muốn tắt.
- Video test (`TEST_VIDEOS`) có thể có annotation nhưng **không được dùng để train hay chỉnh ngưỡng**.

Cấu trúc: `videos/*.mp4` + `annotations.csv`. Chỉ dùng video bạn có quyền dùng, người quay đã đồng ý.
**Giấy phép:** model gốc CC BY-NC 4.0, chỉ phi thương mại.

In [9]:
%pip install -q transformers opencv-python-headless scikit-learn ultralytics lapx

import os, sys, json, random, shutil, time, importlib, csv, pickle
import cv2
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

CLASSES = ["normal", "shoplifting"]

_VEXT = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}
_BASES = [b for b in (os.environ.get("DATA_ROOT"), "/kaggle/input", "/content/drive/MyDrive", ".", "data") if b and Path(b).exists()]

def _find_videos():
    """Thư mục chứa nhiều video nhất (tên thư mục tuỳ ý)."""
    best, n_best = None, 0
    for b in _BASES:
        cnt = defaultdict(int)
        for p in Path(b).rglob("*"):
            if p.suffix.lower() in _VEXT and "work_crops" not in p.parts:
                cnt[p.parent] += 1
        for d, n in cnt.items():
            if n > n_best: best, n_best = d, n
    if best is not None and best.name.lower() in ("normal", "shoplifting"):
        best = best.parent                      # dataset chia sẵn theo thư mục nhãn
    return best or Path("videos")

def _find_csvs():
    """Mọi file annotations*.csv (gộp lại), vd. annotations.csv của bạn + annotations_ucf.csv."""
    out, seen = [], set()
    for b in _BASES:
        for p in sorted(Path(b).rglob("annotations*.csv")):
            r = p.resolve()
            if r not in seen and "work_crops" not in p.parts: seen.add(r); out.append(p)
    return out

# File nhãn chính. Có file này thì CHỈ dùng nó (bỏ các annotations*.csv cũ, vd. "annotations_fixed (2).csv" trong ucf-shopp,
# để 2 bản nhãn không cùng "ghi đè" nhau). ANN_CSV=a.csv,b.csv để dùng nhiều file (vd. thêm annotations_kaggle.csv).
ANN_CSV_DEFAULT = "/kaggle/input/datasets/ltrungphong/annotations-fixed-3/annotations_fixed.csv"
def _ann_csvs():
    want = [s_.strip() for s_ in os.environ.get("ANN_CSV", ANN_CSV_DEFAULT).split(",") if s_.strip()]
    hit = [p for p in want if Path(p).is_file()]
    miss = [p for p in want if p not in hit]
    if miss: print("CẢNH BÁO: không thấy file nhãn:", miss, "(đã Add Input dataset chứa nó chưa?)")
    if hit:
        others = [str(p) for p in _find_csvs() if Path(p).resolve() not in {Path(h).resolve() for h in hit}]
        if others: print("Bỏ qua file nhãn khác vì đã chỉ định ANN_CSV:", others)
        return hit
    print("-> tự dò mọi annotations*.csv trong dữ liệu (cách cũ)")
    return [str(p) for p in _find_csvs()]

K = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
CFG = dict(
    base=os.environ.get("BASE", "MCG-NJU/videomae-base-finetuned-kinetics"),
    videos_dir=str(_find_videos()), ann_csv=_ann_csvs(),
    work_dir=str((Path("/kaggle/temp") if Path("/kaggle/temp").exists() else K) / "work_crops"), out_dir=str(K / "checkpoints" / "videomae_crops_v1"),
    # video KHÔNG BAO GIỜ được train/chỉnh ngưỡng (sửa tên cho đúng file của bạn)
    test_videos=[v for v in os.environ.get("TEST_VIDEOS", "theft_overhead_cctv.mp4").split(",") if v],
    detector=os.environ.get("DETECTOR", "yolo"), yolo_weights="yolov8s.pt", yolo_conf=0.15,
    clip_sec=float(os.environ.get("CLIP_SEC", 3.0)), hop_sec=float(os.environ.get("HOP_SEC", 1.0)),
    margin=0.18, auto_clip_max_sec=float(os.environ.get("AUTO_CLIP_MAX", 15)), mask_others=os.environ.get("MASK", "1") == "1",
    ctx_sec=float(os.environ.get("CTX", 40)), max_range_sec=240.0, neg_range_sec=float(os.environ.get("NEG_RANGE", 90)),   # chỉ phân tích đoạn quanh vụ trộm / đầu video normal
    max_pos=80, max_near_neg=60, max_far_neg=40, neg_every=3,                                                            # giới hạn số cửa sổ train mỗi video
    jpeg_quality=92, src_ratio=float(os.environ.get("SRC_RATIO", 3)), neg_ratio=float(os.environ.get("NEG_RATIO", 4)), boost_max=int(os.environ.get("BOOST_MAX", 4)), select_src=os.environ.get("SELECT_SRC", "auto"), auto_test_pos=int(os.environ.get("AUTO_TEST_POS", 2)), auto_test_neg=int(os.environ.get("AUTO_TEST_NEG", 6)), pos_overlap=0.5, neg_overlap=0.0,   # >=50% cửa sổ nằm trong khoảng trộm -> dương; 0% -> âm; giữa -> bỏ
    val_video_frac=0.25, max_fa_per_min=float(os.environ.get("MAX_FA", 0.2)),
    epochs=int(os.environ.get("EPOCHS", 6)), batch=int(os.environ.get("BATCH", 4)), accum=2,
    lr=float(os.environ.get("LR", 3e-5)), weight_decay=0.05, warmup_frac=0.1,
    grad_ckpt=os.environ.get("GRAD_CKPT", "1") == "1", workers=int(os.environ.get("WORKERS", 2)), seed=0,
    freeze=int(os.environ.get("FREEZE", 4)),            # số block encoder đầu bị đóng băng (0 = mở hết)
    lowres_p=float(os.environ.get("LOWRES_P", 0.3)),    # xác suất giả lập camera độ phân giải thấp khi train
    scene_dhash=float(os.environ.get("SCENE_DHASH", 10)),  # ngưỡng dHash gom clip cùng cảnh (0 = tắt)
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", device, "|", {k: CFG[k] for k in ("videos_dir", "ann_csv", "test_videos", "clip_sec", "detector")})

Note: you may need to restart the kernel to use updated packages.
Bỏ qua file nhãn khác vì đã chỉ định ANN_CSV: ['/kaggle/input/datasets/nsm177/ucf-shopp/annotations_fixed (2).csv']
thiết bị: cuda | {'videos_dir': '/kaggle/input/datasets/nsm177/ucf-shopp/ufc-dataset/ufc-dataset', 'ann_csv': ['/kaggle/input/datasets/ltrungphong/annotations-fixed-3/annotations_fixed.csv'], 'test_videos': ['theft_overhead_cctv.mp4'], 'clip_sec': 3.0, 'detector': 'yolo'}


In [10]:
%%writefile videomae_lite.py
"""Tiện ích VideoMAE cho demo phát hiện hành vi nghi trộm cắp.

Dùng chung cho notebook huấn luyện và script demo thời gian thực:
  - đọc clip từ file video (chỉ giữ các khung cần thiết, tiết kiệm RAM)
  - tiền xử lý (resize, chuẩn hóa) và tăng cường dữ liệu cho huấn luyện
  - Predictor: nạp model đã lưu và trả xác suất "shoplifting" cho một cửa sổ khung hình
"""
from __future__ import annotations

import json
from pathlib import Path

import cv2
import numpy as np
import torch

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}


def uniform_indices(n_total: int, n_frames: int = 16, offset: float = 0.0,
                    span: float = 1.0) -> np.ndarray:
    """n_frames chỉ số đều nhau trong một đoạn con của [0, n_total).

    span   : tỉ lệ độ dài đoạn con so với cả clip (1.0 = cả clip).
    offset : vị trí bắt đầu của đoạn con (0 = đầu, 1 = sát cuối).
    """
    length = max(1, int(round(n_total * span)))
    start = int(round((n_total - length) * offset))
    return np.linspace(start, start + length - 1, n_frames).round().astype(int)


def clip_info(path) -> tuple[int, float]:
    """(số khung hình, fps) của một file video."""
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    cap.release()
    return n, fps


def read_video_frames(path, n_frames: int = 16, size: int = 224, train: bool = False,
                      rng: np.random.Generator | None = None) -> np.ndarray:
    """Đọc n_frames khung hình đều nhau -> uint8 [T, size, size, 3] (RGB).

    Khi train=True lấy ngẫu nhiên một đoạn con 70-100% của clip để đa dạng hóa nhịp độ.
    """
    rng = rng or np.random.default_rng()
    n_total, _ = clip_info(path)
    if n_total <= 0:
        raise ValueError(f"Không đọc được số khung hình: {path}")
    span, offset = (float(rng.uniform(0.7, 1.0)), float(rng.uniform(0, 1))) if train else (1.0, 0.0)
    idx = uniform_indices(n_total, n_frames, offset, span)
    need = set(idx.tolist())
    cap = cv2.VideoCapture(str(path))
    got, last = {}, None
    for i in range(int(idx.max()) + 1):
        if not cap.grab():
            break
        if i in need:
            ok, f = cap.retrieve()
            if ok:
                last = cv2.cvtColor(cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA),
                                    cv2.COLOR_BGR2RGB)
                got[i] = last
    cap.release()
    if last is None:
        raise ValueError(f"Không giải mã được khung nào: {path}")
    return np.stack([got.get(int(i), last) for i in idx])   # thiếu khung -> lặp khung cuối


def augment(frames: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Tăng cường nhẹ, áp dụng giống nhau cho mọi khung của clip."""
    t, h, w, _ = frames.shape
    if rng.random() < 0.5:                                   # lật gương
        frames = frames[:, :, ::-1]
    s = float(rng.uniform(0.8, 1.0))                         # cắt ngẫu nhiên rồi phóng lại
    ch, cw = int(h * s), int(w * s)
    y0, x0 = int(rng.integers(0, h - ch + 1)), int(rng.integers(0, w - cw + 1))
    crop = frames[:, y0:y0 + ch, x0:x0 + cw]
    frames = np.stack([cv2.resize(np.ascontiguousarray(f), (w, h), interpolation=cv2.INTER_LINEAR)
                       for f in crop])
    alpha, beta = float(rng.uniform(0.8, 1.2)), float(rng.uniform(-20, 20))   # tương phản, sáng
    return np.clip(frames.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


def to_tensor(frames: np.ndarray, mean=IMAGENET_MEAN, std=IMAGENET_STD) -> torch.Tensor:
    """uint8 [T,H,W,3] -> float [T,3,H,W] đã chuẩn hóa (đúng dạng pixel_values của VideoMAE)."""
    x = torch.from_numpy(np.ascontiguousarray(frames)).float() / 255.0
    x = (x - torch.tensor(mean)) / torch.tensor(std)
    return x.permute(0, 3, 1, 2).contiguous()


def resize_rgb(frame_bgr: np.ndarray, size: int) -> np.ndarray:
    return cv2.cvtColor(cv2.resize(frame_bgr, (size, size), interpolation=cv2.INTER_AREA),
                        cv2.COLOR_BGR2RGB)


class Predictor:
    """Nạp bundle (model/ + preprocess.json + calibration.json) và chấm điểm một cửa sổ."""

    def __init__(self, bundle_dir, device: str | None = None):
        from transformers import VideoMAEForVideoClassification
        d = Path(bundle_dir)
        self.meta = json.loads((d / "preprocess.json").read_text(encoding="utf-8"))
        self.cal = json.loads((d / "calibration.json").read_text(encoding="utf-8"))
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = VideoMAEForVideoClassification.from_pretrained(d / "model").to(self.device).eval()
        self.pos = int(self.model.config.label2id["shoplifting"])
        self.n_frames = int(self.meta["num_frames"])
        self.size = int(self.meta["size"])
        self.mean, self.std = self.meta["mean"], self.meta["std"]
        self.threshold = float(self.cal["threshold"])
        self.clip_sec = float(self.meta["clip_sec"])

    @torch.no_grad()
    def predict(self, frames_rgb: np.ndarray) -> float:
        """frames_rgb: uint8 [n_frames, size, size, 3]. Trả xác suất 'shoplifting'."""
        x = to_tensor(frames_rgb, self.mean, self.std).unsqueeze(0).to(self.device)
        return float(torch.softmax(self.model(pixel_values=x).logits, -1)[0, self.pos])


Overwriting videomae_lite.py


In [11]:
%%writefile crops_lite.py
"""Cắt vùng vuông quanh từng người và tạo cửa sổ clip cho VideoMAE.

Dùng chung cho notebook huấn luyện và pipeline chạy thật, để lúc train và lúc chạy
tiền xử lý GIỐNG HỆT nhau: bbox người -> thêm lề -> vuông hóa -> resize -> 16 khung đều nhau.
"""
from __future__ import annotations

from collections import defaultdict

import cv2
import numpy as np


# ---------------------------------------------------------------- cắt vuông quanh người
def square_box(xyxy, margin: float = 0.18):
    """(tâm x, tâm y, cạnh) của hình vuông bao quanh bbox, đã cộng lề mỗi phía."""
    x0, y0, x1, y1 = (float(v) for v in xyxy)
    side = max(x1 - x0, y1 - y0) * (1.0 + 2.0 * margin)
    return (x0 + x1) / 2.0, (y0 + y1) / 2.0, max(side, 8.0)


def crop_square(frame_bgr: np.ndarray, xyxy, margin: float = 0.18, size: int = 224) -> np.ndarray:
    """Cắt vuông quanh bbox, phóng về size x size, trả RGB. Phần vượt mép khung lặp lại pixel biên."""
    cx, cy, side = square_box(xyxy, margin)
    s = size / side
    m = np.array([[s, 0.0, size / 2.0 - s * cx], [0.0, s, size / 2.0 - s * cy]], np.float32)
    out = cv2.warpAffine(frame_bgr, m, (size, size), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


def crop_masked(frame_bgr: np.ndarray, target, others, margin: float = 0.18, size: int = 224,
                gray: int = 114) -> np.ndarray:
    """Như crop_square nhưng tô xám vùng của NHỮNG NGƯỜI KHÁC, giữ nguyên bên trong bbox của người đang xét.

    Mục đích: crop quanh một người không còn chứa hành động của người đứng cạnh, nên điểm của
    người này không bị kéo theo người kia. Dùng giống hệt nhau lúc train và lúc chạy thật.
    """
    if others:
        h, w = frame_bgr.shape[:2]
        f = frame_bgr.copy()
        for ob in others:
            x0, y0, x1, y1 = (int(round(v)) for v in ob)
            f[max(0, y0):max(0, min(h, y1)), max(0, x0):max(0, min(w, x1))] = gray
        tx0, ty0, tx1, ty1 = (int(round(v)) for v in target)
        tx0, ty0, tx1, ty1 = max(0, tx0), max(0, ty0), min(w, tx1), min(h, ty1)
        f[ty0:ty1, tx0:tx1] = frame_bgr[ty0:ty1, tx0:tx1]
        frame_bgr = f
    return crop_square(frame_bgr, target, margin, size)


# ---------------------------------------------------------------- bộ phát hiện + theo dõi
class YoloTracker:
    """YOLO (chỉ lớp person) + ByteTrack có giữ trạng thái giữa các khung hình.

    Mỗi video cần reset() để ID bắt đầu lại và bộ lọc Kalman không mang từ video trước sang.
    """

    def __init__(self, weights: str = "yolov8s.pt", conf: float = 0.15,
                 tracker: str = "bytetrack.yaml", imgsz: int = 640, device=None):
        self.weights, self.conf, self.tracker = weights, conf, tracker
        self.imgsz, self.device = imgsz, device
        self.model = None
        self.reset()

    def reset(self):
        """Bắt đầu video mới: ID và Kalman về trạng thái sạch, nhưng chỉ nạp trọng số YOLO một lần."""
        if self.model is None:
            from ultralytics import YOLO
            self.model = YOLO(self.weights)
        try:
            from ultralytics.trackers.basetrack import BaseTrack
            BaseTrack.reset_id()
        except Exception:
            pass
        self.model.predictor = None            # buộc tạo lại tracker ở lần track() kế tiếp

    def update(self, frame_bgr: np.ndarray):
        """-> list[(track_id, (x0, y0, x1, y1), conf)] của khung hiện tại."""
        r = self.model.track(frame_bgr, persist=True, classes=[0], conf=self.conf,
                             tracker=self.tracker, imgsz=self.imgsz, device=self.device,
                             verbose=False)[0]
        if r.boxes is None or r.boxes.id is None:
            return []
        ids = r.boxes.id.int().cpu().tolist()
        xyxy = r.boxes.xyxy.cpu().numpy()
        cf = r.boxes.conf.cpu().numpy()
        return [(int(i), tuple(float(v) for v in b), float(c)) for i, b, c in zip(ids, xyxy, cf)]


class MotionBlobTracker:
    """Bộ theo dõi giả dựa trên trừ nền, CHỈ để kiểm thử mã bằng video tổng hợp. Không dùng thật."""

    def __init__(self, min_area: int = 80):
        self.min_area = min_area
        self.reset()

    def reset(self):
        self.bg = cv2.createBackgroundSubtractorMOG2(history=30, varThreshold=25, detectShadows=False)

    def update(self, frame_bgr: np.ndarray):
        mask = self.bg.apply(frame_bgr)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = [c for c in cnts if cv2.contourArea(c) >= self.min_area]
        if not cnts:
            return []
        x, y, w, h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        return [(1, (float(x), float(y), float(x + w), float(y + h)), 1.0)]


# ---------------------------------------------------------------- quỹ đạo và cửa sổ
def extract_tracks(path, tracker, max_gap_sec: float = 1.5, min_track_sec: float = 1.5,
                   start_f: int = 0, end_f: int | None = None) -> dict:
    """Chạy tracker qua video (hoặc chỉ đoạn khung [start_f, end_f)). Nội suy tuyến tính các khoảng mất phát hiện ngắn.

    Số khung trong tracks là số khung TUYỆT ĐỐI của video. Trả {"fps", "n_frames" (số khung đã phân tích),
    "start_f", "size": (W, H), "tracks": {track_id: {khung: (x0, y0, x1, y1)}}}.
    """
    cap = cv2.VideoCapture(str(path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tracker.reset()
    raw: dict[int, dict[int, tuple]] = defaultdict(dict)
    if start_f > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_f))
    i = int(start_f)
    while True:
        if end_f is not None and i >= end_f:
            break
        ok, frame = cap.read()
        if not ok:
            break
        for tid, box, _ in tracker.update(frame):
            raw[tid][i] = box
        i += 1
    cap.release()
    det_frames = len({f for d in raw.values() for f in d})
    max_gap, tracks = int(round(max_gap_sec * fps)), {}
    for tid, d in raw.items():
        fr = sorted(d)
        if len(fr) < min_track_sec * fps:
            continue
        full = {}
        for a, b in zip(fr[:-1], fr[1:]):
            full[a] = d[a]
            gap = b - a - 1
            if 0 < gap <= max_gap:
                pa, pb = np.array(d[a]), np.array(d[b])
                for k in range(1, gap + 1):
                    full[a + k] = tuple(float(v) for v in (1 - k / (gap + 1)) * pa + (k / (gap + 1)) * pb)
        full[fr[-1]] = d[fr[-1]]
        tracks[tid] = full
    return dict(fps=fps, n_frames=i - int(start_f), start_f=int(start_f), size=(w, h), tracks=tracks, det_frames=det_frames, raw_tracks=len(raw))


def make_windows(info: dict, clip_sec: float, hop_sec: float, n_frames: int = 16,
                 min_cover: float = 0.9) -> list[dict]:
    """Cửa sổ trượt theo từng track. Chỉ giữ cửa sổ mà track có mặt >= min_cover số khung."""
    fps = info["fps"]
    length, hop = max(n_frames, int(round(clip_sec * fps))), max(1, int(round(hop_sec * fps)))
    out = []
    for tid, tr in info["tracks"].items():
        fr = np.array(sorted(tr))
        first, last = int(fr[0]), int(fr[-1])
        starts = list(range(first, last - length + 2, hop))
        if starts and starts[-1] != last - length + 1:
            starts.append(last - length + 1)                       # cửa sổ sát cuối track
        for s in starts:
            if s < 0:
                continue
            inside = np.count_nonzero((fr >= s) & (fr < s + length))
            if inside / length < min_cover:
                continue
            idx = np.linspace(s, s + length - 1, n_frames).round().astype(int)
            boxes, others = [], []
            for j in idx:
                k = fr[np.argmin(np.abs(fr - j))]                    # khung gần nhất còn có bbox
                boxes.append(tr[int(k)])
                others.append([t2[int(j)] for o, t2 in info["tracks"].items() if o != tid and int(j) in t2])
            out.append(dict(track=int(tid), start_f=int(s), end_f=int(s + length),
                            t0=s / fps, t1=(s + length) / fps, idx=idx, boxes=boxes, others=others))
    return out


def render_windows(path, windows: list[dict], size: int = 224, margin: float = 0.18,
                   mask_others: bool = False) -> list[np.ndarray]:
    """Giải mã video đúng một lần, trả mỗi cửa sổ là uint8 [n_frames, size, size, 3] (RGB).

    mask_others=True: tô xám người khác trong crop (xem crop_masked).
    """
    if not windows:
        return []
    n = len(windows[0]["idx"])
    arrays = [np.zeros((n, size, size, 3), np.uint8) for _ in windows]
    need: dict[int, list[tuple[int, int]]] = defaultdict(list)
    for wi, w in enumerate(windows):
        for k, f in enumerate(w["idx"]):
            need[int(f)].append((wi, k))
    cap = cv2.VideoCapture(str(path))
    first, last = min(need), max(need)
    if first > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(first))      # nhảy tới khung đầu tiên cần, khỏi giải mã từ đầu
    for i in range(first, last + 1):
        if not cap.grab():
            break
        if i in need:
            ok, frame = cap.retrieve()
            if not ok:
                continue
            for wi, k in need[i]:
                w = windows[wi]
                arrays[wi][k] = (crop_masked(frame, w["boxes"][k], w["others"][k], margin, size)
                                 if mask_others else crop_square(frame, w["boxes"][k], margin, size))
    cap.release()
    return arrays


# ---------------------------------------------------------------- quy tắc cảnh báo
def alarm_events(times, probs, thr: float, smooth: int = 1, k: int = 1) -> list[float]:
    """Thời điểm (giây) các sự kiện cảnh báo trên MỘT track.

    Điểm làm mượt = trung bình `smooth` cửa sổ gần nhất; báo khi vượt ngưỡng `k` cửa sổ liên tiếp.
    Mỗi chuỗi vượt ngưỡng liên tục chỉ tính một sự kiện.
    """
    probs = np.asarray(probs, float)
    sm = np.array([probs[max(0, i - smooth + 1): i + 1].mean() for i in range(len(probs))])
    events, streak = [], 0
    for t, s in zip(times, sm):
        streak = streak + 1 if s >= thr else 0
        if streak == k:
            events.append(float(t))
    return events


Overwriting crops_lite.py


In [12]:
importlib.invalidate_caches()
for m in ("videomae_lite", "crops_lite"): sys.modules.pop(m, None)
import videomae_lite as vl, crops_lite as cl
from videomae_lite import VIDEO_EXTS, to_tensor, augment

# ---- 1. đọc annotation
def collect_videos():
    """Mọi video dưới các thư mục dữ liệu (tất cả dataset đã Add Input). VIDEO_DIRS=a,b chỉ lấy đường dẫn chứa a hoặc b."""
    only = [s for s in os.environ.get("VIDEO_DIRS", "").split(",") if s]
    seen, out = set(), []
    for base in _BASES:
        for p in Path(base).rglob("*"):
            if p.suffix.lower() not in VIDEO_EXTS: continue
            if any(x in p.parts for x in ("work_crops", "checkpoints")): continue
            r = p.resolve()
            if r in seen or (only and not any(s in str(p) for s in only)): continue
            seen.add(r); out.append(p)
    return sorted(out)
vids = collect_videos()
_by = defaultdict(int)
for p in vids: _by[str(p.parent)] += 1
print("video theo thư mục:", dict(_by))
vdir = Path(CFG["videos_dir"])
for tv in CFG["test_videos"]:                # video test có thể nằm ở dataset/thư mục khác
    if not any(p.name == tv for p in vids):
        for b in _BASES:
            hit = next((q for q in Path(b).rglob(tv) if q.is_file()), None)
            if hit: vids.append(hit); print("Đã thêm video test từ", hit); break
vids = sorted(vids)
if not vids:
    print("Các thư mục đã dò:", _BASES)
    for b in _BASES[:3]:
        print(b, "->", [str(p) for p in list(Path(b).rglob("*"))[:15]])
    raise FileNotFoundError(f"Không thấy video trong {vdir}. Hãy Add Data (Kaggle) / đặt video vào thư mục, hoặc đặt DATA_ROOT.")
# ---- 0. loại video trùng hệt nhau (cùng kích thước + cùng 2 MB đầu), vd. normal-44 = normal-52
import hashlib
def _fp(p):
    with open(p, "rb") as fh:
        return (p.stat().st_size, hashlib.md5(fh.read(2_000_000)).hexdigest())
_seen_fp, _dups = {}, []
for p in list(vids):
    k = _fp(p)
    if k in _seen_fp: _dups.append((p.name, _seen_fp[k].name)); vids.remove(p)
    else: _seen_fp[k] = p
if _dups: print(f"Loại {len(_dups)} video trùng hệt (bản sao, bản gốc):", _dups)
ann = defaultdict(list)                      # tên file -> [(start, end)] các khoảng shoplifting
skip_v = set(); _fixed_vids = set()
ann_pt = {}                                  # (tên file, chỉ số khoảng) -> (px, py): điểm trên NGƯỜI TRỘM tại giây start
vpath = {p.name: p for p in vids}
def _sec(r, k, v):
    """Giây của cột k; nếu trống thì đổi từ cột k_frame theo fps THẬT của video."""
    if (r.get(k) or "").strip(): return float(r[k])
    fr = (r.get(k + "_frame") or "").strip()
    if not fr: return None
    fps = vl.clip_info(vpath[v])[1] if v in vpath else 30.0
    return float(fr) / fps
if CFG["ann_csv"]:
    _isfix = lambda f: Path(f).name.startswith("annotations_fixed")      # file sửa nhãn (có px,py): GHI ĐÈ các dòng cùng video ở file khác
    _fixed_vids |= {r["video"].strip() for f in CFG["ann_csv"] if _isfix(f) for r in csv.DictReader(open(f, encoding="utf-8"))}
    for fcsv in sorted(CFG["ann_csv"], key=lambda f: not _isfix(f)):
        print("đọc nhãn:", fcsv, "(ghi đè)" if _isfix(fcsv) else "")
        for r in csv.DictReader(open(fcsv, encoding="utf-8")):
            if not _isfix(fcsv) and r["video"].strip() in _fixed_vids: continue
            if (r.get("label") or "").strip().lower() == "skip": skip_v.add(r["video"].strip()); continue   # video không rõ nhãn: LOẠI khỏi train/val/test
            if (r.get("label") or "").strip().lower() != "shoplifting": continue
            v = r["video"].strip(); s0, s1 = _sec(r, "start", v), _sec(r, "end", v)
            if s0 is None or s1 is None: print("bỏ dòng thiếu start/end:", r); continue
            ann[v].append((s0, s1))
            if (r.get("px") or "").strip() and (r.get("py") or "").strip():
                ann_pt[(v, len(ann[v]) - 1)] = (float(r["px"]), float(r["py"]))
else:
    print("CẢNH BÁO: không thấy annotations*.csv, mọi video bị coi là normal.")
if skip_v:
    print("LOẠI video (label=skip):", sorted(skip_v)); vids = [p for p in vids if p.name not in skip_v]
skipped = []
for p in vids:                               # dataset kiểu thư mục nhãn (vd. Kaggle kipshidze): không cần CSV
    if p.name not in ann and p.parent.name.lower() == "shoplifting":
        n, fps = vl.clip_info(p)
        if n <= 0: continue
        if n / fps <= CFG["auto_clip_max_sec"]: ann[p.name] = [(0.0, n / fps)]   # clip ngắn: cả clip là trộm
        else: skipped.append(p)                                                # video dài chưa có nhãn giờ: KHÔNG dùng
if skipped:
    sk = {p.resolve() for p in skipped}
    vids = [p for p in vids if p.resolve() not in sk]
    print(f"Bỏ qua {len(skipped)} video trong thư mục shoplifting/ vì dài hơn {CFG['auto_clip_max_sec']:.0f}s và chưa có dòng trong annotations*.csv "
          f"(gán cả video là trộm sẽ sai). Ví dụ: {[p.name for p in skipped[:3]]}")
def _src(p):
    q = p.parts
    if q[:3] == ("/", "kaggle", "input") and len(q) > 4:
        return q[5] if q[3] == "datasets" and len(q) > 6 else q[3]
    return p.parent.name
src_of = {p.name: _src(p) for p in vids}
names = {p.name for p in vids}
for v in list(ann) + CFG["test_videos"]:
    if v not in names: print("CẢNH BÁO: có trong cấu hình/CSV nhưng không có file video:", v)
print(f"{len(vids)} video | {sum(1 for p in vids if p.name in ann)} có đoạn trộm | test cố định: {CFG['test_videos']}")

# ---- 2. phát hiện + theo dõi + tạo cửa sổ cho từng video (có cache)
def make_tracker():
    if CFG["detector"] == "motion":          # chỉ để kiểm thử mã, không dùng thật
        return cl.MotionBlobTracker()
    return cl.YoloTracker(CFG["yolo_weights"], CFG["yolo_conf"])

def pick_primary(info, intervals, vname=""):
    """Mỗi đoạn trộm: track có mặt nhiều khung nhất trong đoạn (người thực hiện hành vi)."""
    prim = {}
    for ai, (a, b) in enumerate(intervals):
        fa, fb = int(a * info["fps"]), int(b * info["fps"])
        pt = ann_pt.get((vname, ai))
        if pt:                               # track có bbox chứa điểm đã chỉ (gần giây start nhất)
            cand = []
            for t, tr in info["tracks"].items():
                fs = [f for f in tr if abs(f - fa) <= int(info["fps"] * 1.5)]
                if not fs: continue
                x0, y0, x1, y1 = tr[min(fs, key=lambda f: abs(f - fa))]
                if x0 <= pt[0] <= x1 and y0 <= pt[1] <= y1:
                    cand.append((abs((x0 + x1) / 2 - pt[0]) + abs((y0 + y1) / 2 - pt[1]), t))
            if cand: prim[ai] = min(cand)[1]; continue
            print(f"CẢNH BÁO {vname}: điểm ({pt[0]:.0f},{pt[1]:.0f}) không nằm trong người nào, dùng cách tự chọn.")
        best = max(info["tracks"], key=lambda t: sum(fa <= f < fb for f in info["tracks"][t]), default=None)
        if best is not None and sum(fa <= f < fb for f in info["tracks"][best]) > 0: prim[ai] = best
    return prim

def label_windows(info, wins, intervals, vname=""):
    prim, fps = pick_primary(info, intervals, vname), info["fps"]
    for w in wins:
        w["y"] = 0
        w["dist"] = min((max(0.0, a - w["t1"], w["t0"] - b2) for a, b2 in intervals), default=99.0)   # giây cách vụ trộm gần nhất (0 = chồng lên)
        if not intervals: continue
        ov = [max(0.0, min(w["t1"], b) - max(w["t0"], a)) / (w["t1"] - w["t0"]) for a, b in intervals]
        best = int(np.argmax(ov))
        if ov[best] >= CFG["pos_overlap"]:
            if prim.get(best) == w["track"]: w["y"] = 1
            else: w["y"] = 0 if (vname, best) in ann_pt else -1   # đã chỉ rõ ai trộm: người khác = âm khó; chưa chỉ: mơ hồ -> bỏ
        elif ov[best] > CFG["neg_overlap"]:
            w["y"] = -1
    return wins

work = Path(CFG["work_dir"]); work.mkdir(parents=True, exist_ok=True)
tracker = make_tracker()
index = {}                                   # tên video -> dict(meta, file)
tag = f"c{CFG['clip_sec']}_h{CFG['hop_sec']}_m{CFG['margin']}_{CFG['detector']}_k{int(CFG['mask_others'])}_x{CFG['ctx_sec']:.0f}_{CFG['neg_range_sec']:.0f}"
def enc(win):                                # [16,224,224,3] RGB -> 16 chuỗi JPEG (nhỏ hơn ~10 lần so với ảnh thô)
    return [cv2.imencode(".jpg", cv2.cvtColor(fr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, CFG["jpeg_quality"]])[1].tobytes() for fr in win]
print(f"cache clip ghi tại {work} | trống {shutil.disk_usage(work).free / 1e9:.1f} GB")
for p in vids:
    f = work / f"{p.stem}__{tag}.pkl"
    meta_f = work / f"{p.stem}__{tag}.json"
    if f.exists() and meta_f.exists():
        meta = json.loads(meta_f.read_text())
    else:
        if shutil.disk_usage(work).free < 1.5e9:
            raise OSError(f"Gần hết dung lượng ở {work} (còn {shutil.disk_usage(work).free/1e9:.1f} GB). Giảm số video, CTX, NEG_RANGE hoặc xoá cache.")
        t0 = time.time()
        n_tot, fps0 = vl.clip_info(p)
        if n_tot <= 0: print("bỏ video không đọc được:", p.name); continue
        dur0, iv = n_tot / fps0, ann.get(p.name, [])
        if iv:                                # chỉ phân tích đoạn quanh vụ trộm (+/- CTX giây), tối đa max_range_sec
            s0 = max(0.0, min(a for a, _ in iv) - CFG["ctx_sec"]); s1 = min(dur0, max(b2 for _, b2 in iv) + CFG["ctx_sec"])
            s1 = min(s1, s0 + CFG["max_range_sec"])
        else:                                 # video normal: chỉ lấy đoạn đầu neg_range_sec giây
            s0, s1 = 0.0, min(dur0, CFG["neg_range_sec"])
        info = cl.extract_tracks(p, tracker, min_track_sec=1.0, start_f=int(s0 * fps0), end_f=int(s1 * fps0))
        wins = cl.make_windows(info, CFG["clip_sec"], CFG["hop_sec"], 16)
        wins = label_windows(info, wins, iv, p.name)
        keep = [w for w in wins if w["y"] >= 0]
        enc_wins = []
        for c in range(0, len(keep), 48):     # chia lô để không tốn RAM
            enc_wins += [enc(x) for x in cl.render_windows(p, keep[c:c + 48], 224, CFG["margin"], CFG["mask_others"])]
        pickle.dump(enc_wins, open(f, "wb"))
        meta = dict(fps=info["fps"], n_frames=info["n_frames"], duration=info["n_frames"] / info["fps"], range=[s0, s1],
                    n_tracks=len(info["tracks"]), det_frames=info["det_frames"], raw_tracks=info["raw_tracks"], ambiguous=sum(w["y"] < 0 for w in wins),
                    windows=[dict(track=w["track"], t0=w["t0"], t1=w["t1"], y=w["y"], dist=w["dist"]) for w in keep])
        meta_f.write_text(json.dumps(meta))
        print(f"{p.name}: phân tích {s0:.0f}-{s1:.0f}s | {meta['n_tracks']} track (thô {meta['raw_tracks']}, {meta['det_frames']}/{meta['n_frames']} khung có người) | {len(keep)} cửa sổ "
              f"(+{sum(w['y']==1 for w in keep)}), bỏ {meta['ambiguous']} mơ hồ | {time.time()-t0:.0f}s | trống {shutil.disk_usage(work).free/1e9:.0f} GB")
    index[p.name] = dict(meta=meta, file=f)

video theo thư mục: {'/kaggle/input/datasets/kipshidze/shoplifting-video-dataset/normal': 90, '/kaggle/input/datasets/kipshidze/shoplifting-video-dataset/shoplifting': 92, '/kaggle/input/datasets/nsm177/ucf-shopp/ufc-dataset/ufc-dataset/Normal': 150, '/kaggle/input/datasets/nsm177/ucf-shopp/ufc-dataset/ufc-dataset/Shoplifting': 50}
Loại 4 video trùng hệt (bản sao, bản gốc): [('normal-51.mp4', 'normal-50.mp4'), ('normal-52.mp4', 'normal-44.mp4'), ('normal-56.mp4', 'normal-55.mp4'), ('Normal_Videos_937_x264.mp4', 'Normal_Videos_936_x264.mp4')]
đọc nhãn: /kaggle/input/datasets/ltrungphong/annotations-fixed-3/annotations_fixed.csv (ghi đè)
Bỏ qua 29 video trong thư mục shoplifting/ vì dài hơn 15s và chưa có dòng trong annotations*.csv (gán cả video là trộm sẽ sai). Ví dụ: ['Shoplifting003_x264.mp4', 'Shoplifting006_x264.mp4', 'Shoplifting008_x264.mp4']
CẢNH BÁO: có trong cấu hình/CSV nhưng không có file video: theft_overhead_cctv.mp4
349 video | 113 có đoạn trộm | test cố định: ['theft_ove

In [14]:
# ---- 3. chia theo VIDEO nguồn (video test cố định không bao giờ vào train/val)
test_v = [v for v in index if v in CFG["test_videos"]]
if not test_v and CFG["auto_test_pos"] + CFG["auto_test_neg"] > 0:       # chưa có video test riêng: tự giữ lại video DÀI chưa dùng
    _r = random.Random(CFG["seed"] + 1)
    _hp = lambda v: any(w["y"] == 1 for w in index[v]["meta"]["windows"])
    # (bản cũ loại video trong annotations_fixed -> test không còn video trộm nào)
    lp = sorted(v for v in index if _hp(v) and index[v]["meta"]["duration"] > 30)
    ln = sorted(v for v in index if not _hp(v) and index[v]["meta"]["duration"] > 30)
    _r.shuffle(lp); _r.shuffle(ln)
    test_v = lp[:CFG["auto_test_pos"]] + ln[:CFG["auto_test_neg"]]
    print("CHƯA có video test riêng (TEST_VIDEOS). Tự giữ lại làm test:", test_v,
          "\n -> chỉ là bước kiểm tra tạm; kết quả thật là video camera của bạn.")
pool = [v for v in index if v not in test_v]
has_pos = lambda v: any(w["y"] == 1 for w in index[v]["meta"]["windows"])
rng = random.Random(CFG["seed"])
pos_v, neg_v = [v for v in pool if has_pos(v)], [v for v in pool if not has_pos(v)]
def _strat(vs):
    """Chọn video val THEO TỪNG NGUỒN (mỗi nguồn có >=2 video thì ít nhất 1 vào val), tránh val toàn clip Kaggle mà không có UCF."""
    g = defaultdict(list)
    for v in vs: g[src_of.get(v, "?")].append(v)
    val = []
    for k, L in g.items():
        rng.shuffle(L)
        if len(L) > 1: val += L[:max(1, round(len(L) * CFG["val_video_frac"]))]
    return val
# ---- nhóm theo CẢNH: clip ngắn cùng camera/bối cảnh/người diễn phải nằm chung một phía (tránh rò rỉ)
def _dhash3(p):
    cap = cv2.VideoCapture(str(p)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); hs = []
    for q in (0.2, 0.5, 0.8):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * q)); ok, im = cap.read()
        if ok:
            g = cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (9, 8)); hs.append((g[:, 1:] > g[:, :-1]).flatten())
    cap.release(); return hs
_par = {v: v for v in index}
def _root(x):
    while _par[x] != x: _par[x] = _par[_par[x]]; x = _par[x]
    return x
if CFG["scene_dhash"] > 0:
    _short = [v for v in pool if index[v]["meta"]["duration"] <= 25 and v in vpath]     # chỉ clip ngắn (bộ Kaggle)
    _h = {v: _dhash3(vpath[v]) for v in _short}
    for i, a in enumerate(_short):
        for b in _short[i + 1:]:
            if len(_h[a]) == 3 and len(_h[b]) == 3 and src_of.get(a) == src_of.get(b) and \
               np.mean([np.count_nonzero(x != y) for x, y in zip(_h[a], _h[b])]) <= CFG["scene_dhash"]:
                _par[_root(a)] = _root(b)
scene_of = {v: _root(v) for v in index}
print(f"nhóm cảnh: {len(set(scene_of[v] for v in pool))} nhóm cho {len(pool)} video train+val")

def _strat_groups(vs):
    """Chọn val theo NHÓM CẢNH trong từng nguồn: cả nhóm vào val hoặc cả nhóm ở lại train.
    Lấy nhóm cho tới khi val có ~val_video_frac số video VÀ số video có trộm của nguồn đó."""
    by_src = defaultdict(lambda: defaultdict(list))
    for v in vs: by_src[src_of.get(v, "?")][scene_of[v]].append(v)
    val = []
    for s, G in by_src.items():
        keys = sorted(G); rng.shuffle(keys)
        if len(keys) < 2: continue
        n_all = sum(len(G[g]) for g in keys); n_pos = sum(has_pos(v) for g in keys for v in G[g])
        got = got_pos = 0
        for g in keys[:-1]:                                  # luôn để lại ít nhất 1 nhóm cho train
            if got >= CFG["val_video_frac"] * n_all and got_pos >= CFG["val_video_frac"] * n_pos: break
            val += G[g]; got += len(G[g]); got_pos += sum(has_pos(v) for v in G[g])
    return val
val_v = _strat_groups(pos_v + neg_v)
train_v = [v for v in pos_v + neg_v if v not in set(val_v)]
print("val theo nguồn:", dict(Counter((src_of.get(v, "?"), has_pos(v)) for v in val_v)))
assert not set(train_v) & set(test_v) and not set(val_v) & set(test_v)

def _cap(L, n): return L if len(L) <= n else [L[int(j)] for j in np.linspace(0, len(L) - 1, n)]
def collect(vs, thin=False):
    """thin=True (chỉ cho TRAIN): giữ mọi cửa sổ dương/âm gần vụ trộm (có giới hạn), và thưa dần cửa sổ âm xa.
    Val/test luôn dùng ĐỦ cửa sổ liên tiếp để luật 'K lần liên tiếp' đo đúng."""
    out = []
    for v in vs:
        W = index[v]["meta"]["windows"]; idx = list(range(len(W)))
        if thin:
            pos = [i for i in idx if W[i]["y"] == 1]
            near = [i for i in idx if W[i]["y"] == 0 and W[i].get("dist", 99) <= 5]
            far = [i for i in idx if W[i]["y"] == 0 and W[i].get("dist", 99) > 5][::CFG["neg_every"]]
            idx = sorted(_cap(pos, CFG["max_pos"]) + _cap(near, CFG["max_near_neg"]) + _cap(far, CFG["max_far_neg"]))
        for i in idx: out.append((v, i, W[i]["y"], W[i]["track"], W[i]["t0"]))
    return out
tr, va, te = collect(train_v, True), collect(val_v), collect(test_v)
def balance_sources(S, ratio):
    """Mỗi nguồn không được có quá `ratio` x số cửa sổ DƯƠNG của nguồn ít dương nhất (chỉ TRAIN).
    Tránh clip ngắn dễ (cả clip là trộm) lấn át video camera dài khó hơn. SRC_RATIO=0 để tắt."""
    if ratio <= 0: return S
    pos = defaultdict(list)
    for s in S:
        if s[2] == 1: pos[src_of.get(s[0], "?")].append(s)
    if len(pos) < 2: return S
    cap = max(20, int(ratio * min(len(v) for v in pos.values())))
    keep = set()
    for k, v in pos.items():
        if len(v) > cap:
            rr = random.Random(CFG["seed"]); v = rr.sample(v, cap)
        keep |= {id(x) for x in v}
        print(f"  nguồn {k}: giữ {len(v)} cửa sổ dương trong train")
    # cửa sổ âm giữ nguyên (không bỏ video camera dài)
    return [s for s in S if s[2] != 1 or id(s) in keep]
def shape_train(S):
    """(1) Mỗi nguồn: số cửa sổ ÂM <= neg_ratio x số cửa sổ DƯƠNG của nguồn đó (giữ âm gần vụ trộm trước) -> hết cảnh hàng nghìn âm UCF 'dìm' vài trăm dương.
       (2) Nhân bản cửa sổ dương của nguồn ít dương hơn (tối đa boost_max lần) để hai nguồn có số dương gần nhau."""
    src = lambda s: src_of.get(s[0], "?")
    P, N = defaultdict(list), defaultdict(list)
    for s in S: (P if s[2] == 1 else N)[src(s)].append(s)
    rr = random.Random(CFG["seed"] + 7); out = []
    for k in set(P) | set(N):
        neg = N[k]
        if CFG["neg_ratio"] > 0 and P[k] and len(neg) > CFG["neg_ratio"] * len(P[k]):
            cap = int(CFG["neg_ratio"] * len(P[k]))
            dist = lambda s: index[s[0]]["meta"]["windows"][s[1]].get("dist", 99)
            near = [s for s in neg if dist(s) <= 5]; far = [s for s in neg if dist(s) > 5]
            rr.shuffle(far); neg = near[:cap] + far[:max(0, cap - len(near))]
            print(f"  nguồn {k}: âm {len(N[k])} -> {len(neg)} (<= {CFG['neg_ratio']:g} x dương {len(P[k])})")
        out += neg
    mx = max((len(v) for v in P.values()), default=0)
    for k, pos in P.items():
        f = max(1, min(CFG["boost_max"], round(mx / max(1, len(pos)))))
        if f > 1: print(f"  nguồn {k}: nhân bản dương x{f} ({len(pos)} -> {len(pos) * f})")
        out += pos * f
    return out
_n0 = len(tr); tr = balance_sources(tr, CFG["src_ratio"]); tr = shape_train(tr)
if len(tr) != _n0: print(f"cân bằng nguồn: train {_n0} -> {len(tr)} cửa sổ")
for name, S, V in (("train", tr, train_v), ("val", va, val_v), ("test", te, test_v)):
    print(f"{name:5s}: {len(V)} video | {len(S)} cửa sổ | dương {sum(s[2]==1 for s in S)} | âm {sum(s[2]==0 for s in S)}")
if sum(s[2] == 1 for s in tr) < 30:
    print("CẢNH BÁO: dưới 30 cửa sổ dương trong train. Cần nhiều video trộm hơn; kết quả sẽ rất nhiễu.")
if not any(s[2] == 1 for s in va): print("CẢNH BÁO: val không có cửa sổ dương, không chọn được ngưỡng đáng tin.")
if not any(s[2] == 1 for s in te): print("CẢNH BÁO: test KHÔNG có cửa sổ trộm -> không đo được khả năng bắt trộm. Đặt TEST_VIDEOS hoặc ghi thêm nhãn UCF.")
_leak = {scene_of[v] for v in train_v} & {scene_of[v] for v in val_v}
print("nhóm cảnh dùng chung train/val:", len(_leak), "(phải là 0)")

_cache = {}
def get_x(v):
    if v not in _cache:
        if len(_cache) > 3: _cache.pop(next(iter(_cache)))
        _cache[v] = pickle.load(open(index[v]["file"], "rb"))
    return _cache[v]
def decode(win):                              # 16 chuỗi JPEG -> uint8 [16,224,224,3] RGB
    return np.stack([cv2.cvtColor(cv2.imdecode(np.frombuffer(b_, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB) for b_ in win])

CHƯA có video test riêng (TEST_VIDEOS). Tự giữ lại làm test: ['Shoplifting027_x264.mp4', 'Shoplifting015_x264.mp4', 'Normal_Videos_935_x264.mp4', 'Normal_Videos_781_x264.mp4', 'Normal_Videos_868_x264.mp4', 'Normal_Videos_871_x264.mp4', 'Normal_Videos_924_x264.mp4', 'Normal_Videos_783_x264.mp4'] 
 -> chỉ là bước kiểm tra tạm; kết quả thật là video camera của bạn.
nhóm cảnh: 209 nhóm cho 341 video train+val
val theo nguồn: {('shoplifting-video-dataset', False): 17, ('shoplifting-video-dataset', True): 32, ('ucf-shopp', True): 6, ('ucf-shopp', False): 35}
  nguồn shoplifting-video-dataset: giữ 282 cửa sổ dương trong train
  nguồn ucf-shopp: giữ 94 cửa sổ dương trong train
  nguồn ucf-shopp: âm 2776 -> 376 (<= 4 x dương 94)
  nguồn ucf-shopp: nhân bản dương x3 (94 -> 282)
cân bằng nguồn: train 3735 -> 1252 cửa sổ
train: 251 video | 1252 cửa sổ | dương 564 | âm 688
val  : 90 video | 5245 cửa sổ | dương 366 | âm 4879
test : 8 video | 2242 cửa sổ | dương 11 | âm 2231
nhóm cảnh dùng chung trai

In [15]:
from transformers import VideoMAEForVideoClassification
model = VideoMAEForVideoClassification.from_pretrained(
    CFG["base"], num_labels=2, id2label={0: "normal", 1: "shoplifting"},
    label2id={"normal": 0, "shoplifting": 1}, ignore_mismatched_sizes=True).to(device)
if CFG["grad_ckpt"] and device == "cuda": model.gradient_checkpointing_enable()
if CFG["freeze"] > 0:                         # đóng băng embedding + các block đầu: ít dữ liệu nên chỉ tinh chỉnh phần trên
    for p_ in model.videomae.embeddings.parameters(): p_.requires_grad = False
    for blk in model.videomae.encoder.layer[:CFG["freeze"]]:
        for p_ in blk.parameters(): p_.requires_grad = False
print("tham số được train:", sum(p_.numel() for p_ in model.parameters() if p_.requires_grad) / 1e6, "triệu")
N_FR, SIZE = int(model.config.num_frames), int(model.config.image_size)
assert (N_FR, SIZE) == (16, 224), "crop được cắt 16 khung x 224px; model phải khớp"
try:
    from transformers import VideoMAEImageProcessor
    _p = VideoMAEImageProcessor.from_pretrained(CFG["base"]); MEAN, STD = list(_p.image_mean), list(_p.image_std)
except Exception:
    MEAN, STD = list(vl.IMAGENET_MEAN), list(vl.IMAGENET_STD)

class WinDS(Dataset):
    def __init__(self, S, train): self.S, self.train, self.calls = S, train, 0
    def __len__(self): return len(self.S)
    def __getitem__(self, i):
        v, wi, y, _, _ = self.S[i]
        fr = decode(get_x(v)[wi])
        if self.train:
            self.calls += 1                          # num_workers=0 nên bộ đếm này tái lập được
            rng = np.random.default_rng((CFG["seed"] * 1_000_003 + i * 7919 + self.calls) % 2**32)
            fr = augment(np.ascontiguousarray(fr), rng)
            if rng.random() < CFG["lowres_p"]:       # giả lập camera 320x240 (UCF): thu nhỏ rồi phóng lại
                s_ = int(rng.integers(56, 113))
                fr = np.stack([cv2.resize(cv2.resize(f_, (s_, s_), interpolation=cv2.INTER_AREA), (SIZE, SIZE),
                                          interpolation=cv2.INTER_LINEAR) for f_ in fr])
        return to_tensor(fr, MEAN, STD), int(y)

def loader(S, train):
    return DataLoader(WinDS(S, train), batch_size=CFG["batch"], shuffle=train, num_workers=0)  # cache theo video nằm trong tiến trình chính

@torch.no_grad()
def predict_probs(S):
    model.eval(); P = []
    for x, _ in loader(S, False):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            lg = model(pixel_values=x.to(device)).logits
        P.append(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(P) if P else np.zeros(0)

def safe_auc(y, p): return float(roc_auc_score(y, p)) if len(set(y.tolist())) == 2 else float("nan")
def auc_by_source(S, P):
    out = {}
    for s in sorted({src_of.get(x[0], "?") for x in S}):
        ii = [i for i, x in enumerate(S) if src_of.get(x[0], "?") == s]
        y = np.array([S[i][2] for i in ii]); out[s] = dict(auc=safe_auc(y, P[ii]), pos=int(y.sum()), neg=int(len(y) - y.sum()))
    return out
def select_score(S, P):
    """Điểm chọn checkpoint: AUC trung bình trên các nguồn có CẢ dương lẫn âm (thường là UCF = camera thật),
    không phải AUC gộp bị clip Kaggle dễ làm phồng. SELECT_SRC=<tên nguồn> để ép; không có nguồn hợp lệ thì dùng AUC gộp."""
    a = auc_by_source(S, P)
    ok = {k: v["auc"] for k, v in a.items() if v["pos"] and v["neg"] and not np.isnan(v["auc"]) and (CFG["select_src"] in ("auto", k))}
    return (float(np.mean(list(ok.values()))), a) if ok else (safe_auc(np.array([s[2] for s in S]), P), a)
print(f"model {N_FR} khung x {SIZE}px")

config.json:   0%|          | 0.00/22.9k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


tham số được train: 56.699906 triệu


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

model 16 khung x 224px


In [16]:
# ---- huấn luyện
ytr = np.array([s[2] for s in tr]); yva = np.array([s[2] for s in va])
cnt = np.bincount(ytr, minlength=2).astype(float)
class_w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
print("trọng số lớp:", class_w.tolist())
opt = torch.optim.AdamW([p_ for p_ in model.parameters() if p_.requires_grad], lr=CFG["lr"], weight_decay=CFG["weight_decay"])
spe = max(1, len(loader(tr, True)) // CFG["accum"])
total, warm = CFG["epochs"] * spe, max(1, int(CFG["warmup_frac"] * CFG["epochs"] * spe))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
crit = nn.CrossEntropyLoss(weight=class_w)
best, best_state, hist, t0 = -1.0, None, [], time.time()
for ep in range(1, CFG["epochs"] + 1):
    model.train(); run = n = 0; opt.zero_grad()
    for step, (x, y) in enumerate(loader(tr, True), 1):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            loss = crit(model(pixel_values=x).logits.float(), y) / CFG["accum"]
        scaler.scale(loss).backward()
        if step % CFG["accum"] == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        run += loss.item() * CFG["accum"] * len(y); n += len(y)
    pv = predict_probs(va); auc_all = safe_auc(yva, pv); auc_v, a_src = select_score(va, pv)
    hist.append(dict(epoch=ep, train_loss=run / n, val_auc_all=auc_all, val_auc=auc_v, by_source=a_src))
    print(f"epoch {ep:2d} | loss {run/n:.4f} | val AUC gộp {auc_all:.3f} | AUC chọn ckpt (nguồn khó) {auc_v:.3f} | {time.time()-t0:.0f}s")
    print("      theo nguồn:", {k: (round(v['auc'], 3) if not np.isnan(v['auc']) else None, v['pos'], v['neg']) for k, v in a_src.items()})
    score = auc_v if not np.isnan(auc_v) else -0.5 + ep * 1e-3
    if score > best: best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); model.eval()

trọng số lớp: [0.9098837375640869, 1.109929084777832]
epoch  1 | loss 0.4728 | val AUC gộp 0.921 | AUC chọn ckpt (nguồn khó) 0.871 | 420s
      theo nguồn: {'shoplifting-video-dataset': (0.896, 288, 252), 'ucf-shopp': (0.847, 78, 4627)}
epoch  2 | loss 0.1916 | val AUC gộp 0.889 | AUC chọn ckpt (nguồn khó) 0.804 | 883s
      theo nguồn: {'shoplifting-video-dataset': (0.873, 288, 252), 'ucf-shopp': (0.734, 78, 4627)}
epoch  3 | loss 0.1175 | val AUC gộp 0.901 | AUC chọn ckpt (nguồn khó) 0.833 | 1356s
      theo nguồn: {'shoplifting-video-dataset': (0.864, 288, 252), 'ucf-shopp': (0.802, 78, 4627)}
epoch  4 | loss 0.0672 | val AUC gộp 0.913 | AUC chọn ckpt (nguồn khó) 0.844 | 1828s
      theo nguồn: {'shoplifting-video-dataset': (0.871, 288, 252), 'ucf-shopp': (0.817, 78, 4627)}
epoch  5 | loss 0.0507 | val AUC gộp 0.910 | AUC chọn ckpt (nguồn khó) 0.839 | 2299s
      theo nguồn: {'shoplifting-video-dataset': (0.871, 288, 252), 'ucf-shopp': (0.806, 78, 4627)}
epoch  6 | loss 0.0408 | val

VideoMAEForVideoClassification(
  (videomae): VideoMAEModel(
    (embeddings): VideoMAEEmbeddings(
      (patch_embeddings): VideoMAEPatchEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (encoder): VideoMAEEncoder(
      (layer): ModuleList(
        (0-11): 12 x VideoMAELayer(
          (attention): VideoMAEAttention(
            (attention): VideoMAESelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): VideoMAESelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): VideoMAEIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
      

In [17]:
# ---- đánh giá THEO VIDEO: bắt được vụ trộm không, báo nhầm bao nhiêu lần / phút
def per_track_scores(S, probs):
    d = defaultdict(lambda: defaultdict(list))                 # video -> track -> [(t0, p)]
    for s, p in zip(S, probs): d[s[0]][s[3]].append((s[4], float(p)))
    return d

def video_report(vs, S, probs, thr, smooth, k, tol=2.0):
    """Cần S gồm cả cửa sổ mơ hồ? Không: chỉ cửa sổ có nhãn. Báo nhầm = sự kiện ngoài khoảng trộm ± tol."""
    sc = per_track_scores(S, probs); caught = missed = fa = 0; minutes = 0.0; rows = []
    for v in vs:
        iv = ann.get(v, []); dur = index[v]["meta"]["duration"]; ev_all = []
        for tid, lst in sc[v].items():
            lst.sort(); ev_all += cl.alarm_events([a + CFG["clip_sec"] for a, _ in lst], [b for _, b in lst], thr, smooth, k)
        inside = lambda t, a, b: a - tol <= t <= b + tol + CFG["clip_sec"]
        n_fa = sum(not any(inside(t, a, b) for a, b in iv) for t in ev_all)
        det = [any(inside(t, a, b) for t in ev_all) for a, b in iv]
        caught += sum(det); missed += len(det) - sum(det); fa += n_fa
        free = dur - sum(b - a + 2 * tol for a, b in iv); minutes += max(free, 0) / 60
        rows.append((v, len(iv), sum(det), n_fa))
    return dict(caught=caught, missed=missed, false_alarms=fa, normal_minutes=minutes,
                fa_per_min=fa / max(minutes, 1e-9), rows=rows)

# chọn (ngưỡng, làm mượt, K) trên VAL: bắt nhiều nhất với báo nhầm/phút <= mức cho phép
pv = predict_probs(va)
print("AUC val theo NGUỒN (quan trọng hơn AUC gộp):", json.dumps(auc_by_source(va, pv), ensure_ascii=False))
dump = lambda S, P: [dict(video=x[0], src=src_of.get(x[0], "?"), track=int(x[3]), t0=float(x[4]), y=int(x[2]), p=float(p)) for x, p in zip(S, P)]
dump_val, dump_test = dump(va, pv), []
grid = [(t, s, k) for s in (1, 2, 3) for k in (1, 2, 3) for t in np.round(np.arange(0.3, 0.96, 0.05), 2) if k <= s + 1]
best_rule, best_key = None, None
for t, s, k in grid:
    r = video_report(val_v, va, pv, t, s, k)
    ok = r["fa_per_min"] <= CFG["max_fa_per_min"]
    key = (ok, r["caught"], -r["fa_per_min"], -t)
    if best_key is None or key > best_key: best_key, best_rule = key, (float(t), s, k)
thr, smooth, kk = best_rule
rv = video_report(val_v, va, pv, thr, smooth, kk)
print(f"quy tắc chọn trên val: ngưỡng {thr}, làm mượt {smooth} cửa sổ, {kk} liên tiếp | val bắt {rv['caught']}/{rv['caught']+rv['missed']}, "
      f"{rv['fa_per_min']:.2f} báo nhầm/phút")
if not (best_key[0]): print("CẢNH BÁO: không có quy tắc nào đạt mức báo nhầm cho phép trên val.")
if len(val_v) < 3: print("CẢNH BÁO: val chỉ có", len(val_v), "video, quy tắc rất có thể chỉ khớp riêng val.")

# --- KẾT QUẢ CUỐI: video test chưa từng dùng
metrics = {}
if te:
    pt = predict_probs(te); yt = np.array([s[2] for s in te])
    dump_test = dump(te, pt)
    print("AUC test theo nguồn:", json.dumps(auc_by_source(te, pt), ensure_ascii=False))
    rt = video_report(test_v, te, pt, thr, smooth, kk)
    metrics = dict(test_window_auc=safe_auc(yt, pt), test_caught=rt["caught"], test_missed=rt["missed"],
                   test_false_alarms=rt["false_alarms"], test_normal_minutes=rt["normal_minutes"],
                   test_fa_per_min=rt["fa_per_min"], rule=dict(threshold=thr, smooth=smooth, k=kk))
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    for v, nint, nd, nfa in rt["rows"]: print(f"  {v}: bắt {nd}/{nint} đoạn trộm, {nfa} báo nhầm")
    for v in test_v:
        sc = per_track_scores(te, pt)[v]
        for tid, lst in sc.items():
            ps = [p for _, p in sorted(lst)]
            if ps: print(f"  {v} track {tid}: đỉnh {max(ps):.2f} | trung bình {np.mean(ps):.2f}")
    print("Lưu ý: vài video test = vài con số, không phải bằng chứng. Càng ít video càng đừng tin.")
else:
    print("Không có video test trong dữ liệu; chưa có đánh giá cuối cùng.")

AUC val theo NGUỒN (quan trọng hơn AUC gộp): {"shoplifting-video-dataset": {"auc": 0.8959711199294532, "pos": 288, "neg": 252}, "ucf-shopp": {"auc": 0.8468160684499565, "pos": 78, "neg": 4627}}
quy tắc chọn trên val: ngưỡng 0.75, làm mượt 2 cửa sổ, 2 liên tiếp | val bắt 30/39, 0.11 báo nhầm/phút
AUC test theo nguồn: {"ucf-shopp": {"auc": 0.39238417342406584, "pos": 11, "neg": 2231}}
{
  "test_window_auc": 0.39238417342406584,
  "test_caught": 0,
  "test_missed": 3,
  "test_false_alarms": 1,
  "test_normal_minutes": 10.216666666666667,
  "test_fa_per_min": 0.09787928221859707,
  "rule": {
    "threshold": 0.75,
    "smooth": 2,
    "k": 2
  }
}
  Shoplifting027_x264.mp4: bắt 0/2 đoạn trộm, 0 báo nhầm
  Shoplifting015_x264.mp4: bắt 0/1 đoạn trộm, 0 báo nhầm
  Normal_Videos_935_x264.mp4: bắt 0/0 đoạn trộm, 0 báo nhầm
  Normal_Videos_781_x264.mp4: bắt 0/0 đoạn trộm, 1 báo nhầm
  Normal_Videos_868_x264.mp4: bắt 0/0 đoạn trộm, 0 báo nhầm
  Normal_Videos_871_x264.mp4: bắt 0/0 đoạn trộm, 0 báo

In [18]:
# ---- lưu bundle (khớp pipeline chạy thật)
out = Path(CFG["out_dir"]); out.mkdir(parents=True, exist_ok=True)
model.save_pretrained(out / "model")
json.dump(dict(num_frames=N_FR, size=SIZE, mean=MEAN, std=STD, clip_sec=CFG["clip_sec"], base=CFG["base"],
               classes=CLASSES, person_crop=True, crop_square=True, crop_margin=CFG["margin"], mask_others=CFG["mask_others"],
               hop_sec=CFG["hop_sec"], detector=dict(weights=CFG["yolo_weights"], conf=CFG["yolo_conf"], tracker="bytetrack", classes=[0]),
               note="Cắt vuông quanh từng người (lề margin mỗi phía); mask_others=true thì tô xám người khác (crops_lite.crop_masked); 16 khung đều nhau trong clip_sec giây, resize 224."),
          open(out / "preprocess.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump(dict(threshold=thr, smooth_windows=smooth, consecutive_k=kk, max_fa_per_min_target=CFG["max_fa_per_min"],
               note="Điểm = trung bình smooth cửa sổ gần nhất của MỘT track; báo khi vượt ngưỡng k lần liên tiếp."),
          open(out / "calibration.json", "w", encoding="utf-8"), indent=2)
json.dump(dict(model="VideoMAE fine-tune person-crop (2 lớp)", base=CFG["base"],
               license_note="Model gốc CC BY-NC 4.0: chỉ dùng phi thương mại", metrics_test=metrics, history=hist,
               split=dict(train_videos=sorted(train_v), val_videos=sorted(val_v), test_videos=sorted(test_v)),
               windows=dict(train=len(tr), val=len(va), test=len(te)), annotations={k: v for k, v in ann.items()},
               duplicates_removed=_dups, freeze=CFG["freeze"], lowres_p=CFG["lowres_p"], scene_groups=len(set(scene_of.values())),
               trained_at=time.strftime("%Y-%m-%d %H:%M")),
          open(out / "meta.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump(dict(clip_sec=CFG["clip_sec"], hop_sec=CFG["hop_sec"], val=dump_val, test=dump_test), open(out / "window_scores.json", "w", encoding="utf-8"))
for f in ("videomae_lite.py", "crops_lite.py"): shutil.copy(f, out / f)
print("Đã lưu bundle:", out.resolve())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu bundle: /kaggle/working/checkpoints/videomae_crops_v1


## Dùng bundle trong pipeline

`preprocess.json` ghi rõ cách cắt (`clip_sec`, `crop_margin`, vuông hóa) và `calibration.json` ghi quy tắc báo động (ngưỡng, làm mượt, K liên tiếp).
Pipeline chạy thật phải dùng **đúng** `crops_lite.crop_square` và các thông số đó, nếu không sẽ lại lệch phân phối như lần trước.
Cảnh báo chỉ nghĩa là "nghi vấn, cần người xem lại".

In [19]:
import shutil, os
from IPython.display import FileLink

B = "/kaggle/working/checkpoints/videomae_crops_v1"
print(os.listdir(B))                                   # kiểm tra: phải có model, preprocess.json, calibration.json...

shutil.make_archive("/kaggle/working/videomae_bundle", "zip", B)
shutil.rmtree("/kaggle/working/work_crops", ignore_errors=True)   # xoá ảnh cắt tạm (~vài GB)
print("zip:", round(os.path.getsize("/kaggle/working/videomae_bundle.zip") / 1e6), "MB")

FileLink("videomae_bundle.zip")

['preprocess.json', 'model', 'videomae_lite.py', 'crops_lite.py', 'calibration.json', 'window_scores.json', 'meta.json']
zip: 281 MB


/kaggle/working/videomae_bundle.zip